<a href="https://colab.research.google.com/github/viettungvuong/HR-Information/blob/main/HR_Information.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install datasets

In [11]:
from datasets import load_dataset

dataset_name = "tinixai/vietnamese-job-descriptions"
dataset = load_dataset(dataset_name)

df = dataset['train'].to_pandas()
print(df.head())

   id                                          job_title  \
0   1                Java Software Engineer - MSCV 233-2   
1   2  Software Engineer (Backend/Frontend) - Tiếng T...   
2   3  Lead Software Engineer – Desktop (Remote, Engl...   
3   4              Lâp Trình Viên/ Software Engineer/ IT   
4   5                        Software Engineer (Fintech)   

                                        company_name         salary  \
0                       CÔNG TY TNHH QUICK VIỆT NAM  26 - 36 triệu   
1                          CÔNG TY TNHH SONIC FUSION  15 - 35 triệu   
2  CÔNG TY TNHH PHẦN MỀM GIÁ TRỊ TÍNH TOÁN VÀ ỨNG...       4000 usd   
3  CÔNG TY TNHH ĐẦU TƯ THƯƠNG MẠI VÀ DỊCH VỤ PHÁT...  10 - 15 triệu   
4                                 CÔNG TY TNHH CASSO  14 - 20 triệu   

      location        job_type job_industry experience_level education_level  \
0  Hồ Chí Minh  Toàn thời gian  IT phần mềm            3 năm        Cao đẳng   
1  Hồ Chí Minh  Toàn thời gian  IT phần mềm       Dư

### Step 1: Entity Extraction Strategy
To build the Knowledge Graph, you'll iterate through your dataset and pass the unstructured text to an LLM to extract structured entities and relationships. Here is an example of how you would format the prompt for a single job row.

In [14]:
# Let's take the requirements from the second job in our dataset as an example
sample_job_id = df.iloc[1]['id']
sample_requirement = df.iloc[1]['requirements']

print(f"--- Original Requirement Text (Job ID: {sample_job_id}) ---")
print(sample_requirement)

# This is the prompt template you would send to an LLM (like Gemini or OpenAI)
prompt_template = f"""
You are an expert Knowledge Graph extractor specializing in HR and recruitment.
Analyze the following job requirement text and extract the required skills, programming languages, and tools.

Return the output strictly as a JSON list of relationships, like this:
[{{"source": "Job_{sample_job_id}", "relationship": "REQUIRES_SKILL", "target": "Python"}}]

Text:
{sample_requirement}
"""

print("\n--- Prompt to send to LLM ---")
print(prompt_template)

--- Original Requirement Text (Job ID: 2) ---
Thành thạo ít nhất một trong các ngôn ngữ: Python / Java / NodeJS / PHP Ưu điểm: có kinh nghiệm về AI, Machine Learning hoặc AdTech, đã làm việc với hệ thống data lớn hoặc automation Có khả năng đọc hiểu tài liệu tiếng Trung (HSK4+ là lợi thế) Có kinh nghiệm lập trình từ 1 năm trở lên Có hiểu biết về API, database, hệ thống backend

--- Prompt to send to LLM ---

You are an expert Knowledge Graph extractor specializing in HR and recruitment.
Analyze the following job requirement text and extract the required skills, programming languages, and tools.

Return the output strictly as a JSON list of relationships, like this:
[{"source": "Job_2", "relationship": "REQUIRES_SKILL", "target": "Python"}]

Text:
Thành thạo ít nhất một trong các ngôn ngữ: Python / Java / NodeJS / PHP Ưu điểm: có kinh nghiệm về AI, Machine Learning hoặc AdTech, đã làm việc với hệ thống data lớn hoặc automation Có khả năng đọc hiểu tài liệu tiếng Trung (HSK4+ là lợi thế)

### Step 2: Knowledge Graph
Let's use Neo4j to extract dataset into a knowledge graph since it's structured.

In [9]:
pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 17.7 MB/s eta 0:00:00


In [15]:
from neo4j import GraphDatabase

# 1. Connection Details
URI = "neo4j+s://48099f3d.databases.neo4j.io"
AUTH = ("48099f3d", "Vt2o8fQLxuRgsUN0Bf-4WaRv4zvdM7hMEqCsUMBWgUg")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [18]:
def inject_vietnamese_jobs(uri, auth, dataframe, rows_limit=1000):
    # 1. Limit the data immediately
    # We slice the dataframe (or records) to the first 'limit' rows
    records = dataframe.head(rows_limit).to_dict(orient='records')

    print(f"Starting injection for {len(records)} records...")

    with GraphDatabase.driver(uri, auth=auth) as driver:
        # Create Constraints
        driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (j:Job) REQUIRE j.id IS UNIQUE")
        driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE")
        driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.name IS UNIQUE")

        query = """
        UNWIND $batch AS row
        MERGE (j:Job {id: row.id})
        SET j.title = row.job_title,
            j.salary = row.salary,
            j.experience = row.experience_level,
            j.description = row.job_description

        WITH j, row WHERE row.company_name IS NOT NULL
        MERGE (c:Company {name: row.company_name})
        MERGE (j)-[:POSTED_BY]->(c)

        WITH j, row WHERE row.location IS NOT NULL
        MERGE (l:Location {name: row.location})
        MERGE (j)-[:LOCATED_IN]->(l)
        """

        # 2. Process the limited set in batches
        batch_size = 500
        for i in range(0, len(records), batch_size):
            batch = records[i:i + batch_size]
            driver.execute_query(query, batch=batch)
            print(f"Injected {i + len(batch)} / {len(records)} records...", end='\r')

    print("\nInjection complete.")



In [19]:
# To run it for just the first 100 rows:
inject_vietnamese_jobs(URI, AUTH, df, rows_limit=100)

Starting injection for 100 records...
Injected 100 / 100 records...
Injection complete.


### Step 2: Data Cleaning
Clean data to extract more information

In [20]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.1 MB/s eta 0:00:00


In [27]:
import os
import json
from groq import Groq
from google.colab import userdata

# Initialize Groq client securely using Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=groq_api_key)

def extract_structured_data(benefits_text, requirements_text):
    prompt = f"""
    You are an expert HR data structurer. Extract the benefits and requirements from the provided text into clear bullet points.

    CRITICAL RULES:
    1. Return a JSON object with exactly two keys: "benefits" (a list of strings) and "requirements" (a list of strings).
    2. For "benefits", if salary or compensation information is mentioned, it MUST be the very first item in the "benefits" list. Anything related to SHUI (Bảo hiểm xã hội) must be second. All of those must be noted with (Salary) and (Social Insurance) tags at the end of the bullet point.
    3. Keep the extracted bullet points concise and in Vietnamese if the original text is in Vietnamese.

    Benefits Text:
    {benefits_text}

    Requirements Text:
    {requirements_text}
    """

    try:
        response = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful data extraction assistant that outputs strictly valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="openai/gpt-oss-20b",
            response_format={"type": "json_object"},
            temperature=0.1
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error during Groq API call: {e}")
        return {"benefits": [], "requirements": []}


In [26]:
# Let's test the extraction on the first 3 rows of our dataset
sample_df = df.head(3).copy()

print("Extracting structured data using Groq...\n")
for index, row in sample_df.iterrows():
    print(f"--- Job ID: {row['id']} | {row['job_title']} ---")

    # Call our Groq function
    extracted = extract_structured_data(row['benefits'], row['requirements'])

    print("\nRequirements:")
    for req in extracted.get('requirements', []):
        print(f" - {req}")

    print("\nBenefits:")
    for ben in extracted.get('benefits', []):
        print(f" - {ben}")

    print("="*60 + "\n")

Extracting structured data using Groq...

--- Job ID: 1 | Java Software Engineer - MSCV 233-2 ---

Requirements:
 - Từ 25 ~ 39 tuổi
 - Trên 3 năm kinh nghiệm lập trình web: JavaScript, jQuery, Spring/Struts/Seasar2/Hibernate, Java, PostgreSQL/MySQL
 - Khả năng research giải quyết vấn đề kỹ thuật
 - Kinh nghiệm DevOps: Git, GitHub, Jenkins, Docker
 - Khả năng tạo/trình bày tài liệu kỹ thuật dễ hiểu, chia sẻ kiến thức
 - Kinh nghiệm Automation test: Jmeter, JUnit, Selenium
 - Ưu tiên: Hiểu quy trình làm việc tại công ty Nhật
 - Sử dụng hiệu quả công cụ AI: Github Copilot, Gemini, Claude, ChatGPT
 - Đọc hiểu tài liệu tiếng Anh tốt
 - Khả năng giao tiếp tốt
 - Khả năng đọc, hiểu, phân tích yêu cầu nghiệp vụ khách hàng
 - Khả năng phát hiện và giải quyết vấn đề tốt

Benefits:
 - Lương: 1,000 ~ 1,400 USD (Gross) (Salary)
 - Bảo Hiểm xã hội đóng trên Full lương Hợp đồng (Social Insurance)
 - Du lịch Công ty hàng năm, tài trợ 50% chi phí tour cho gia đình nhân viên
 - Lương tháng 13+14
 - Nhiề

In [ ]:
import re

def extract_and_normalize_salary(text):
    """
    Extracts salary numbers and currencies from text and normalizes them.
    Handles formats like: '15 - 35 triệu', '4000 usd', '1000 ~ 1400 USD', '26 - 36 triệu'
    """
    if not text or not isinstance(text, str):
        return None

    text = text.lower().replace(',', '')

    # Regex for range format: 'min - max currency' or 'min ~ max currency'
    range_pattern = r'(\d+(?:\.\d+)?)\s*(?:-|~|đến)\s*(\d+(?:\.\d+)?)\s*(triệu|tr|usd|\$|vnd|vnđ)'

    # Regex for single value: 'value currency'
    single_pattern = r'(\d+(?:\.\d+)?)\s*(triệu|tr|usd|\$|vnd|vnđ)'

    range_match = re.search(range_pattern, text)
    if range_match:
        min_val = float(range_match.group(1))
        max_val = float(range_match.group(2))
        currency = range_match.group(3)
        return normalize_value(min_val, currency), normalize_value(max_val, currency), currency

    single_match = re.search(single_pattern, text)
    if single_match:
        val = float(single_match.group(1))
        currency = single_match.group(2)
        norm_val = normalize_value(val, currency)
        return norm_val, norm_val, currency

    return None

def normalize_value(value, currency):
    """
    Normalizes the value based on the currency string.
    Assumes base unit is VND. USD conversion is approx (can be adjusted).
    """
    currency = currency.lower()
    if currency in ['triệu', 'tr']:
        return value * 1_000_000
    elif currency in ['usd', '$']:
        return value * 25_000 # Example conversion rate
    else:
        return value

# Test the extraction
sample_salaries = [
    '15 - 35 triệu',
    '4000 usd',
    '1,000 ~ 1,400 USD',
    '26 - 36 triệu',
    'Lên đến 50 triệu VNĐ'
]

for s in sample_salaries:
    print(f"Original: '{s}' -> Extracted/Normalized (Min, Max, Currency): {extract_and_normalize_salary(s)}")